In [4]:
import numpy as np
import matplotlib.pyplot as plt                         
import matplotlib.patches as patches
import seaborn as sns
import scipy.signal as signal 
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.model_selection import cross_val_score
from sklearn.metrics import roc_curve, auc
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from sklearn.metrics import accuracy_score
import pandas as pd

In [5]:
df = pd.read_csv('EEG_data.csv')
df.head()

,SubjectID,VideoID,Attention,Mediation,Raw,Delta,Theta,Alpha1,Alpha2,Beta1,Beta2,Gamma1,Gamma2,predefinedlabel,user-definedlabeln
0,0.0,0.0,56.0,43.0,278.0,301963.0,90612.0,33735.0,23991.0,27946.0,45097.0,33228.0,8293.0,0.0,0.0
1,0.0,0.0,40.0,35.0,-50.0,73787.0,28083.0,1439.0,2240.0,2746.0,3687.0,5293.0,2740.0,0.0,0.0
2,0.0,0.0,47.0,48.0,101.0,758353.0,383745.0,201999.0,62107.0,36293.0,130536.0,57243.0,25354.0,0.0,0.0
3,0.0,0.0,47.0,57.0,-5.0,2012240.0,129350.0,61236.0,17084.0,11488.0,62462.0,49960.0,33932.0,0.0,0.0
4,0.0,0.0,44.0,53.0,-8.0,1005145.0,354328.0,37102.0,88881.0,45307.0,99603.0,44790.0,29749.0,0.0,0.0


In [6]:
# split into usable x and y
feats = ['Raw','Delta','Theta','Alpha1','Alpha2','Beta1','Beta2','Gamma1','Gamma2']
x_unprocessed = df[['SubjectID','VideoID','Raw','Delta','Theta','Alpha1','Alpha2','Beta1','Beta2','Gamma1','Gamma2','predefinedlabel']]
grouped = x_unprocessed.groupby(['SubjectID','VideoID']).mean()
X = grouped[feats]
y = grouped['predefinedlabel'] #1 = confused, 2=not confused

In [7]:
#finding all subsets of all features
sublist_feats = [a[i:j] for i in range(len(a)) for j in range(i + 1, len(a) + 1)]  

In [8]:
# train test split, 60/20/20 train/val/test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.25, random_state=42)

testing SVM

In [9]:
from sklearn.svm import SVC

In [14]:
#baseline
SVM_model = SVC()
SVM_model.fit(X_train,y_train)
val_pred = SVM_model.predict(X_val)
print(f'f1_score:{f1_score(y_val, val_pred)}')
print(f'accuracy:{accuracy_score(y_val, val_pred)}')

f1_score:0.7333333333333333
accuracy:0.6


In [49]:
#testing which features
out = pd.DataFrame(columns=['features', 'f1', 'accuracy'])
f1 = []
accuracies = []
for sublist in sublist_feats:
    cur_X_train = X_train[sublist]
    cur_X_val = X_val[sublist]
    cur_SVM = SVC()
    cur_SVM.fit(cur_X_train,y_train)
    preds = cur_SVM.predict(cur_X_val)
    f1.append(f1_score(y_val, preds))
    accuracies.append(accuracy_score(y_val,preds))
out = pd.DataFrame({'features': sublist_feats,'f1':f1,'accuracy':accuracies})
best_feats = out.sort_values(['f1','accuracy'],ascending=False).iloc[0]
print(best_feats)

features    [Delta, Theta]
f1                0.740741
accuracy              0.65
Name: 10, dtype: object


In [51]:
# final test
SVM_model = SVC()
SVM_model.fit(X_train[best_feats.features],y_train)
test_pred = SVM_model.predict(X_test[best_feats.features])
print(f'f1_score:{f1_score(y_test, test_pred)}')
print(f'accuracy:{accuracy_score(y_test, test_pred)}')

f1_score:0.4166666666666667
accuracy:0.3


testing XGBoost

In [41]:
import xgboost as xgb

In [42]:
xgb_clf = xgb.XGBClassifier(use_label_encoder=False, eval_metric='logloss')
xgb_clf.fit(X_train, y_train)
y_pred = xgb_clf.predict(X_test)

/home/jancayan/.local/lib/python3.11/site-packages/xgboost/training.py:183: UserWarning: [23:59:18] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


In [43]:
#testing which features
out = pd.DataFrame(columns=['features', 'f1', 'accuracy'])
f1 = []
accuracies = []
for sublist in sublist_feats:
    cur_X_train = X_train[sublist]
    cur_X_val = X_val[sublist]
    cur_xgb = xgb.XGBClassifier(use_label_encoder=False, eval_metric='logloss')
    cur_xgb.fit(cur_X_train,y_train)
    preds = cur_xgb.predict(cur_X_val)
    f1.append(f1_score(y_val, preds))
    accuracies.append(accuracy_score(y_val,preds))
out = pd.DataFrame({'features': sublist_feats,'f1':f1,'accuracy':accuracies})
best_feats = out.sort_values(['f1','accuracy'],ascending=False).iloc[0]
print(best_feats)

/home/jancayan/.local/lib/python3.11/site-packages/xgboost/training.py:183: UserWarning: [23:59:22] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/home/jancayan/.local/lib/python3.11/site-packages/xgboost/training.py:183: UserWarning: [23:59:22] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/home/jancayan/.local/lib/python3.11/site-packages/xgboost/training.py:183: UserWarning: [23:59:22] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/home/jancayan/.local/lib/python3.11/site-packages/xgboost/training.py:183: UserWarning: [23:59:22] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/home/jancayan/.local/lib/python3.11/site-packages/xgboo

features    [Theta, Alpha1, Alpha2, Beta1]
f1                                0.782609
accuracy                              0.75
Name: 20, dtype: object


/home/jancayan/.local/lib/python3.11/site-packages/xgboost/training.py:183: UserWarning: [23:59:23] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/home/jancayan/.local/lib/python3.11/site-packages/xgboost/training.py:183: UserWarning: [23:59:23] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/home/jancayan/.local/lib/python3.11/site-packages/xgboost/training.py:183: UserWarning: [23:59:23] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/home/jancayan/.local/lib/python3.11/site-packages/xgboost/training.py:183: UserWarning: [23:59:23] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/home/jancayan/.local/lib/python3.11/site-packages/xgboo

In [53]:
# final test
xgb_clf = xgb.XGBClassifier(use_label_encoder=False, eval_metric='logloss')
xgb_clf.fit(X_train[best_feats.features], y_train)
y_pred = xgb_clf.predict(X_test[best_feats.features])
print(f'f1_score:{f1_score(y_test, y_pred)}')
print(f'accuracy:{accuracy_score(y_test, y_pred)}')

f1_score:0.4
accuracy:0.4


/home/jancayan/.local/lib/python3.11/site-packages/xgboost/training.py:183: UserWarning: [00:02:21] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Random Forest

In [56]:
from sklearn.ensemble import RandomForestClassifier

In [57]:
rf_clf = RandomForestClassifier(random_state=42)
rf_clf.fit(X_train, y_train)
y_pred = rf_clf.predict(X_test)

In [58]:
#testing which features
out = pd.DataFrame(columns=['features', 'f1', 'accuracy'])
f1 = []
accuracies = []
for sublist in sublist_feats:
    cur_X_train = X_train[sublist]
    cur_X_val = X_val[sublist]
    cur_f = RandomForestClassifier(random_state=42)
    cur_f.fit(cur_X_train,y_train)
    preds = cur_f.predict(cur_X_val)
    f1.append(f1_score(y_val, preds))
    accuracies.append(accuracy_score(y_val,preds))
out = pd.DataFrame({'features': sublist_feats,'f1':f1,'accuracy':accuracies})
best_feats = out.sort_values(['f1','accuracy'],ascending=False).iloc[0]
print(best_feats)

features    [Theta, Alpha1, Alpha2, Beta1]
f1                                0.782609
accuracy                              0.75
Name: 20, dtype: object


In [59]:
# final test
rf_clf = RandomForestClassifier(random_state=42)
rf_clf.fit(X_train[best_feats.features], y_train)
y_pred = rf_clf.predict(X_test[best_feats.features])
print(f'f1_score:{f1_score(y_test, y_pred)}')
print(f'accuracy:{accuracy_score(y_test, y_pred)}')

f1_score:0.5882352941176471
accuracy:0.65
